[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-02-first-flow.ipynb#scrollTo=1a2b3c4d)

---
# Day 2 · Your First Flow — YAML Structure and Hello World
**certified-journeys / kestra-certified** · Day 2 · Flow Authoring

> **Goal for today:** Author a complete Kestra flow YAML from scratch, understand every required field, chain tasks in sequence, and be able to export/import a flow between namespaces.

In [ ]:
%pip install -q pyyaml requests

## Step 1 · The Minimal Valid Flow — Three Required Fields

A Kestra flow needs exactly **three required top-level fields**:

| Field | Type | Purpose |
|-------|------|--------|
| `id` | string | Unique name within namespace (kebab-case) |
| `namespace` | string | Hierarchical grouping (dot-separated) |
| `tasks` | list | At least one task object |

Everything else — `description`, `labels`, `inputs`, `triggers`, `variables` — is optional.

The simplest meaningful flow has one `Log` task:

In [ ]:
import yaml

# The absolute minimum: id + namespace + one task
hello_world_flow = {
    "id": "hello-world",
    "namespace": "tutorial",
    "tasks": [
        {
            "id": "greet",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Hello, Kestra!"
        }
    ]
}

print("=== Minimal Hello World Flow ===")
print(yaml.dump(hello_world_flow, default_flow_style=False, sort_keys=False))

# Verify the three required fields are present
required = ["id", "namespace", "tasks"]
missing = [f for f in required if f not in hello_world_flow]
print(f"Required fields present: {not missing}")
print(f"Flow unique key: {hello_world_flow['namespace']}.{hello_world_flow['id']}")

### What just happened?
- **`id` + `namespace` is the composite unique key** — two flows in different namespaces can share the same `id`.
- The `Log` task (`io.kestra.plugin.core.log.Log`) is Kestra's built-in print statement — no extra plugin needed.
- `yaml.dump` with `sort_keys=False` preserves field order — important because Kestra reads `id` before `tasks`.
- Paste this YAML into the Kestra UI editor and click **Save** — it will immediately appear in your flow list.

## Step 2 · Adding Description and Labels for Discoverability

In a real team environment, flows need metadata so others can discover and understand them:

| Field | Purpose | Example |
|-------|---------|--------|
| `description` | Free-text explanation shown in the UI | `"Ingests raw CSV files from S3"` |
| `labels` | Key-value tags for filtering | `{team: data-eng, env: prod}` |
| `disabled` | Pause a flow without deleting it | `true` |
| `revision` | Auto-incremented on each save | managed by Kestra |

Labels are indexed — you can filter the flows list by `team:data-eng` in the Kestra UI search bar.

In [ ]:
# Add discoverability metadata to our hello-world flow
documented_flow = {
    "id": "hello-world",
    "namespace": "tutorial",
    "description": "|",  # placeholder — we'll use a multiline string below
    "labels": {
        "team": "data-engineering",
        "env": "development",
        "day": "02"
    },
    "tasks": [
        {
            "id": "greet",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Hello, Kestra!"
        }
    ]
}

# Proper multiline description
documented_flow["description"] = (
    "A Hello World flow for learning Kestra basics.\n"
    "It logs a greeting message and exits successfully.\n"
    "Part of the certified-journeys kestra-certified course."
)

print(yaml.dump(documented_flow, default_flow_style=False, sort_keys=False))

# Show how labels help filter in large teams
flow_catalogue = [
    {"id": "hello-world",    "namespace": "tutorial",        "labels": {"team": "data-engineering", "env": "development"}},
    {"id": "daily-etl",      "namespace": "company.data",    "labels": {"team": "data-engineering", "env": "production"}},
    {"id": "send-report",    "namespace": "company.finance", "labels": {"team": "finance",          "env": "production"}},
    {"id": "clean-warehouse","namespace": "company.data",    "labels": {"team": "data-engineering", "env": "production"}},
]

# Filter by label — simulates Kestra UI search
def filter_flows(catalogue, **label_filters):
    return [
        f for f in catalogue
        if all(f["labels"].get(k) == v for k, v in label_filters.items())
    ]

prod_de_flows = filter_flows(flow_catalogue, team="data-engineering", env="production")
print("\nData-engineering production flows:")
for f in prod_de_flows:
    print(f"  {f['namespace']}.{f['id']}")

### What just happened?
- **Labels are just strings** — Kestra doesn't enforce any schema, so agree on conventions with your team.
- A common convention: `team`, `env`, `project`, `owner` labels for every flow.
- **`description` supports Markdown** — Kestra renders it in the UI flow detail pane.
- Our `filter_flows` function mirrors what `GET /api/v1/flows?labels=team:data-engineering` does via the API.

## Step 3 · Chaining Tasks in Sequence

Tasks in `tasks[]` execute **sequentially by default** — each task waits for the previous one to succeed.

To run tasks in **parallel**, you use a `Parallel` task group (Day 4 topic). For now, let's understand sequential flow:

```
start → extract → validate → transform → load → end
   ↑ each arrow = wait for previous to complete
```

Each task can also pass outputs to the next via `{{ outputs.taskId.key }}`.

In [ ]:
# A sequential 4-task pipeline with data passing between tasks
sequential_flow = {
    "id": "sequential-etl",
    "namespace": "tutorial",
    "description": "Four-step ETL demonstrating sequential task execution",
    "tasks": [
        {
            "id": "extract",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Step 1: Extracting 1000 rows from source database"
        },
        {
            "id": "validate",
            "type": "io.kestra.plugin.core.log.Log",
            # References the previous task's output (simulated here — in real Kestra
            # this would be the actual output of the extract task)
            "message": "Step 2: Validating extracted data — source: {{ outputs.extract.executionId }}"
        },
        {
            "id": "transform",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Step 3: Applying business transformations"
        },
        {
            "id": "load",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Step 4: Loading to data warehouse"
        }
    ]
}

print(yaml.dump(sequential_flow, default_flow_style=False, sort_keys=False))

# Simulate execution order
print("\nExecution graph (sequential):")
tasks = sequential_flow["tasks"]
for i, task in enumerate(tasks):
    arrow = "→ " if i > 0 else "  "
    print(f"  {arrow}[{i+1}] {task['id']} ({task['type'].split('.')[-1]})")

### What just happened?
- Tasks execute in **array order** — the YAML position IS the execution order.
- If any task fails, **subsequent tasks are skipped** and the execution state becomes `FAILED`.
- `{{ outputs.taskId.executionId }}` is one of the built-in output variables — the Log task is minimal but shows the pattern.
- **Topology view** in the Kestra UI draws arrows between tasks automatically from this array order.

## Step 4 · Understanding Task Properties

Every Kestra task shares a **common set of properties** regardless of its `type`:

| Property | Required | Type | Purpose |
|----------|----------|------|---------|
| `id` | ✅ | string | Unique within the flow; used in output references |
| `type` | ✅ | string | Fully-qualified plugin class name |
| `description` | ❌ | string | Shown in topology view tooltip |
| `timeout` | ❌ | duration | Kill task if it exceeds this duration (e.g. `PT5M`) |
| `retry` | ❌ | object | Retry policy: `type`, `maxAttempts`, `delay` |
| `allowFailure` | ❌ | boolean | Continue flow even if this task fails |
| `disabled` | ❌ | boolean | Skip this task without removing it |

Duration format uses ISO 8601: `PT5M` = 5 minutes, `PT1H30M` = 90 minutes.

In [ ]:
# A task with all common properties populated
robust_task = {
    "id": "fetch-external-api",
    "type": "io.kestra.plugin.core.http.Request",
    "description": "Fetches daily exchange rates from a public API",
    "timeout": "PT2M",                   # kill if it takes > 2 minutes
    "allowFailure": False,               # propagate failure to stop the flow
    "retry": {
        "type": "exponential",
        "maxAttempts": 3,
        "interval": "PT30S",            # base wait: 30 seconds
        "maxInterval": "PT5M",          # cap at 5 minutes
        "multiplier": 2.0               # double wait each time: 30s → 60s → 120s
    },
    # Task-specific property (HTTP plugin)
    "uri": "https://api.exchangerate.host/latest?base=USD",
    "method": "GET"
}

# Build a flow using this robust task
robust_flow = {
    "id": "exchange-rate-monitor",
    "namespace": "tutorial",
    "tasks": [robust_task]
}

print(yaml.dump(robust_flow, default_flow_style=False, sort_keys=False))

# Parse the retry config to show the wait schedule
retry = robust_task["retry"]
base_s = 30
print("\nRetry wait schedule (exponential):")
wait = base_s
for attempt in range(1, retry["maxAttempts"] + 1):
    cap = 5 * 60  # maxInterval = PT5M
    actual = min(wait, cap)
    print(f"  Attempt {attempt}: wait {actual}s before retry")
    wait = int(wait * retry["multiplier"])

### What just happened?
- **Retry policies** are configured directly in the task YAML — no code needed.
- `exponential` retry type doubles the wait each attempt until `maxInterval` is reached.
- `timeout: PT2M` uses ISO 8601 duration format — `P` = period, `T` = time part, `2M` = 2 minutes.
- **`allowFailure: true`** lets you mark optional tasks (like sending a Slack notification) so a failure there doesn't abort your main pipeline.

## Step 5 · Flow Inputs — Parameterising Your Flows

Kestra flows can declare typed **inputs** that users supply at execution time. This avoids hardcoding values in the YAML.

Input types: `STRING`, `INT`, `FLOAT`, `BOOLEAN`, `DATETIME`, `DATE`, `TIME`, `DURATION`, `FILE`, `JSON`, `ARRAY`, `URI`

In [ ]:
# A parameterised flow with typed inputs
parameterised_flow = {
    "id": "parameterised-etl",
    "namespace": "tutorial",
    "inputs": [
        {
            "id": "source_table",
            "type": "STRING",
            "description": "Source table name to extract",
            "defaults": "orders"          # optional default value
        },
        {
            "id": "row_limit",
            "type": "INT",
            "description": "Maximum rows to process",
            "defaults": 10000
        },
        {
            "id": "dry_run",
            "type": "BOOLEAN",
            "description": "If true, skip the load step",
            "defaults": False
        }
    ],
    "tasks": [
        {
            "id": "extract",
            "type": "io.kestra.plugin.core.log.Log",
            # Inputs are accessed via {{ inputs.inputId }}
            "message": "Extracting up to {{ inputs.row_limit }} rows from {{ inputs.source_table }}"
        },
        {
            "id": "load",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Loading data to warehouse (dry_run={{ inputs.dry_run }})",
            "disabled": False   # in production you'd use an If task to skip on dry_run
        }
    ]
}

print(yaml.dump(parameterised_flow, default_flow_style=False, sort_keys=False))

# Simulate resolving inputs at execution time
def simulate_execution(flow: dict, **input_values) -> dict:
    """Show what the flow looks like with inputs resolved."""
    import re
    # Build effective inputs (merge defaults + overrides)
    effective = {}
    for inp in flow.get("inputs", []):
        effective[inp["id"]] = input_values.get(inp["id"], inp.get("defaults"))
    return effective

# Default run
defaults = simulate_execution(parameterised_flow)
print("\nDefault inputs:", defaults)

# Custom run
custom = simulate_execution(parameterised_flow, source_table="customers", row_limit=500, dry_run=True)
print("Custom inputs: ", custom)

### What just happened?
- **Inputs with defaults** make flows user-friendly — the UI shows a form with pre-filled values.
- Kestra validates input types at submission time — submitting a string for an `INT` input fails immediately.
- `{{ inputs.inputId }}` works anywhere in the YAML — in task properties, trigger configs, even `namespace`.
- For conditional skipping (like `dry_run`), use `io.kestra.plugin.core.flow.If` — covered in Day 5.

## Step 6 · Exporting and Importing Flows Between Namespaces

Kestra flows are just YAML — exporting means downloading the YAML; importing means `PUT`-ing it back.

Common use case: **promote a flow from `dev` to `prod`** by changing the namespace.

The Kestra API:
- **Export:** `GET /api/v1/flows/{namespace}/{id}` — returns the YAML
- **Import/Update:** `PUT /api/v1/flows` — body is the YAML (Content-Type: application/x-yaml)
- **Bulk export:** `GET /api/v1/flows/export/by-namespace?namespace=tutorial` — returns a ZIP

In [ ]:
# Simulate promoting a flow from 'tutorial' namespace to 'company.data.engineering'

def promote_flow(flow: dict, target_namespace: str, version_label: str = None) -> dict:
    """
    Promote a flow to a different namespace.
    Optionally add a version label.
    Returns a NEW dict — does not mutate the original.
    """
    import copy
    promoted = copy.deepcopy(flow)
    promoted["namespace"] = target_namespace

    # Update environment label
    if "labels" not in promoted:
        promoted["labels"] = {}
    promoted["labels"]["env"] = "production"
    promoted["labels"]["promoted-from"] = flow["namespace"]

    if version_label:
        promoted["labels"]["version"] = version_label

    return promoted

# Original dev flow
dev_flow = {
    "id": "sequential-etl",
    "namespace": "tutorial",
    "labels": {"env": "development", "team": "data-engineering"},
    "tasks": [
        {"id": "extract",   "type": "io.kestra.plugin.core.log.Log", "message": "Extracting…"},
        {"id": "transform", "type": "io.kestra.plugin.core.log.Log", "message": "Transforming…"},
        {"id": "load",      "type": "io.kestra.plugin.core.log.Log", "message": "Loading…"}
    ]
}

prod_flow = promote_flow(dev_flow, "company.data.engineering", version_label="v1.2.0")

print("=== DEV flow ===")
print(yaml.dump(dev_flow, default_flow_style=False, sort_keys=False))

print("=== PROMOTED to PROD ===")
print(yaml.dump(prod_flow, default_flow_style=False, sort_keys=False))

# Show the curl command to import into Kestra
prod_yaml = yaml.dump(prod_flow, default_flow_style=False, sort_keys=False)
print("\nTo import into a running Kestra instance:")
print("  curl -X PUT http://localhost:8080/api/v1/flows \\")
print("       -H 'Content-Type: application/x-yaml' \\")
print("       --data-binary @flow.yaml")

### What just happened?
- **Promotion is just a namespace change** — the flow `id` stays the same, the namespace changes.
- This is a key advantage of YAML-first: promotion is a `sed` command or a one-liner Python function.
- In CI/CD, your pipeline exports YAML from dev, runs validation, then `PUT`s to production Kestra.
- **Flow revisions** are tracked automatically — Kestra keeps the full history; you can roll back from the UI.

## Step 7 · Generating Multiple Flow Variants

Because flows are data, you can use Python to generate a family of similar flows without repeating yourself — the YAML equivalent of a DAG factory.

In [ ]:
# Generate one flow per data source — a common pattern in data engineering

data_sources = [
    {"name": "orders",    "schedule": "0 * * * *",   "priority": "high"},   # every hour
    {"name": "customers", "schedule": "0 6 * * *",   "priority": "medium"}, # 6am daily
    {"name": "products",  "schedule": "0 0 * * 1",   "priority": "low"},    # weekly monday
]

def build_ingest_flow(source: dict, namespace: str = "company.data") -> dict:
    """Generate a standardised ingest flow for a given data source."""
    return {
        "id": f"ingest-{source['name']}",
        "namespace": namespace,
        "description": f"Scheduled ingestion of {source['name']} data from source system",
        "labels": {
            "source": source["name"],
            "priority": source["priority"],
            "team": "data-engineering"
        },
        "triggers": [
            {
                "id": f"schedule-{source['name']}",
                "type": "io.kestra.plugin.core.trigger.Schedule",
                "cron": source["schedule"]
            }
        ],
        "tasks": [
            {
                "id": "extract",
                "type": "io.kestra.plugin.core.log.Log",
                "message": f"Extracting {source['name']} data"
            },
            {
                "id": "load",
                "type": "io.kestra.plugin.core.log.Log",
                "message": f"Loading {source['name']} to warehouse"
            }
        ]
    }

generated_flows = [build_ingest_flow(s) for s in data_sources]

print(f"Generated {len(generated_flows)} flows:\n")
for flow in generated_flows:
    trigger_cron = flow["triggers"][0]["cron"]
    print(f"  {flow['namespace']}.{flow['id']}")
    print(f"    schedule: {trigger_cron} | priority: {flow['labels']['priority']}")
    print()

# Print YAML for the first generated flow
print("\n=== Generated YAML for first flow ===")
print(yaml.dump(generated_flows[0], default_flow_style=False, sort_keys=False))

### What just happened?
- We replaced a DAG factory pattern with a simple Python function that outputs YAML strings.
- **Three flows, zero repetition** — changes to the template propagate to all sources automatically.
- In production, this script would write the YAML files and your CI/CD would `PUT` them to Kestra.
- Kestra also has a **CI/CD GitHub Action** (`kestra-io/deploy-action`) that automates this import step.

In [ ]:
# Challenge: Build a complete Kestra flow YAML for a "data-quality-check" flow
#
# Requirements:
#   - id: "data-quality-check"
#   - namespace: "company.data.quality"
#   - description: a meaningful one-line description
#   - labels: team="data-engineering", env="production", priority="high"
#   - 1 input: STRING input called "table_name" with default value "orders"
#   - 3 tasks in sequence:
#       1. row-count-check   — Log: "Checking row count for {{ inputs.table_name }}"
#       2. null-check        — Log: "Checking for nulls in critical columns"
#       3. freshness-check   — Log: "Verifying data freshness — last updated within 24h"
#   - A daily schedule trigger at 07:00 UTC
#   - Validate your flow (use validate_kestra_flow from Day 1 — reproduced below)
#   - Print the YAML

import re

def validate_kestra_flow(flow: dict) -> list:
    """Quick validator — same as Day 1."""
    errors = []
    for field in ["id", "namespace", "tasks"]:
        if field not in flow:
            errors.append(f"Missing required field: '{field}'")
    if "id" in flow and not re.match(r'^[a-z0-9][a-z0-9-]*$', str(flow["id"])):
        errors.append(f"Flow id must be kebab-case")
    if "tasks" in flow and isinstance(flow["tasks"], list):
        seen = set()
        for i, t in enumerate(flow["tasks"]):
            for k in ["id", "type"]:
                if k not in t:
                    errors.append(f"Task {i} missing '{k}'")
            if "id" in t:
                if t["id"] in seen:
                    errors.append(f"Duplicate task id: '{t['id']}'")
                seen.add(t["id"])
    return errors

# Your solution here:
quality_flow = {
    "id": "data-quality-check",
    # TODO: add namespace, description, labels, inputs, triggers, tasks
}

errors = validate_kestra_flow(quality_flow)
if errors:
    print("Validation errors:", errors)
else:
    print("✓ Flow is valid!")
    print(yaml.dump(quality_flow, default_flow_style=False, sort_keys=False))

---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| Required fields | `id`, `namespace`, `tasks` — the flow won't save without them |
| Unique key | `namespace.id` — same `id` can exist in different namespaces |
| Labels | Key-value tags; use for team/env/project; filterable in UI and API |
| Sequential execution | Tasks run in array order; failure stops subsequent tasks |
| Task common props | `timeout`, `retry`, `allowFailure`, `disabled` — on every task type |
| Inputs | Typed parameters; accessed via `{{ inputs.name }}`; can have defaults |
| Promote = namespace swap | Change `namespace` to promote dev → prod; import via `PUT /api/v1/flows` |
| Flow factory | Python generates YAML for families of similar flows |

> **Tip:** Every Kestra flow is a YAML file. The `id` + `namespace` pair is the unique key — you can have 100 flows named `daily-etl` as long as they're in different namespaces.

---
## What's next
**Day 3** → Explore the Kestra plugin library — run Shell scripts, fetch HTTP APIs, and execute Python tasks with automatic dependency management.

Mark Day 2 complete in your [tracker](../index.html).